# Breast tumor classification with logistic regression

This notebook teaches a full machine-learning workflow using the public Wisconsin Diagnostic Breast Cancer dataset included with scikit-learn. The goal is to classify the recorded label as malignant or benign from 30 numeric cell-nucleus measurements. It is an educational benchmark, not a clinical tool.

**Learning path:** define the task → inspect the data → preserve the positive class → split → fit a pipeline → evaluate → discuss limits.

For exact comments and implementation, see [`src/analyze.py`](../src/analyze.py). Run this notebook from the project root after installing `requirements.txt`.

## 1. Set up the environment

Install the dependency ranges with `python -m pip install -r requirements.txt`. The data loader uses the copy bundled with scikit-learn, so it does not require a MIMIC account or a manual data download.

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Find the project root when Jupyter was launched from the project folder
# or its notebooks/ folder. In Colab, clone the whole repository so the
# canonical, commented analysis script is available beside this notebook.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'src' / 'analyze.py').exists() and (PROJECT_DIR.parent / 'src' / 'analyze.py').exists():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / 'src' / 'analyze.py').exists():
    PROJECT_DIR = Path('/content/AI_In_Healthcare_Project')
    if not (PROJECT_DIR / 'src' / 'analyze.py').exists():
        subprocess.run(['git', 'clone', 'https://github.com/vivek0586/AI_In_Healthcare_Project.git', str(PROJECT_DIR)], check=True)
assert (PROJECT_DIR / 'src' / 'analyze.py').exists(), 'Make the complete repository available before running the notebook.'
print('Project:', PROJECT_DIR.resolve())

## 2. Inspect the dataset and define the positive class

Each row is one sample; each column is a numeric measurement. Scikit-learn stores benign as target 1 and malignant as target 0. We recode malignant as 1 so positive-class recall means sensitivity for malignant samples. Always check class encodings instead of guessing.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
malignant_code = list(data.target_names).index('malignant')
y = (data.target == malignant_code).astype(int)
print(f'Samples: {len(X)} | features: {X.shape[1]}')
print('Target (1 = malignant):\n', y.value_counts().rename(index={0: 'benign', 1: 'malignant'}))
display(X.head())

## 3. Run the documented analysis

The canonical implementation lives in `src/analyze.py`. It uses a fixed random seed, a stratified 80/20 holdout, a `StandardScaler` inside a `Pipeline`, and logistic regression. Keeping scaling inside the pipeline prevents the test set from influencing learned preprocessing values. Running the script also saves predictions, metrics, coefficients, and plots.

In [ ]:
# Run the exact same implementation used from the command line.
import runpy
runpy.run_path(str(PROJECT_DIR / 'src' / 'analyze.py'), run_name='__main__')

## 4. Read the held-out results

Sensitivity (recall) is the share of malignant samples detected. Specificity is the share of benign samples correctly classified. Precision is the share of model-positive samples that are malignant in this test sample. AUC measures ranking over thresholds; it does not tell us which threshold is clinically appropriate. Inspect error counts as well as summary metrics.

In [ ]:
metrics_path = PROJECT_DIR / 'outputs' / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
pd.Series(metrics, name='value').to_frame()

## 5. Inspect errors and model coefficients

The confusion matrix distinguishes false negatives from false positives. The coefficient table ranks standardized logistic-regression weights by absolute size. These weights describe this fitted model; they are not causal effects or medical explanations.

Open `outputs/confusion_matrix.png`, `outputs/roc_curve.png`, and `outputs/coefficients.csv` to inspect the generated artifacts.

## 6. Replication and limitations

Re-run `python src/analyze.py` and compare `outputs/metrics.json`. With the same code, random seed, and compatible dependency versions, the split and metrics should reproduce. Different package versions can cause small changes.

This benchmark is small and historical, uses processed measurements from digitized aspirate images, and is not representative evidence for current clinical practice. It does not test external validation, calibration, subgroup performance, workflow effects, or patient outcomes. Do not use the model for care. See `REPLICATION_REPORT.md` for the rerun audit and student checklist.